In [ ]:
from ultralytics import YOLO
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndimage
from scipy.ndimage import gaussian_filter1d

In [ ]:
model=YOLO("/content/drive/MyDrive/best(1).pt").model
file_path = "/content/2017-04-20-Braunschweig-Frankfurt--mitte-4K0G0259--patch41.jpg"
image = cv2.imread(file_path)
image = cv2.resize(image,(640,640))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image = torch.from_numpy(image).permute(2, 0, 1).unsqueeze(0).float() / 255.0

index=torch.topk(model(image)[0][0][5][:], 200).indices[0]

In [ ]:
x_center,y_center,width,height = model(image)[0][0][0:4,undex]
x1 = x_center - width/2
y1 = y_center - height/2
x2 = x_center + width/2
y2 = y_center + height/2
img=cv2.imread(file_path)
cv2.rectangle(img,(int(x1),int(y1)),(int(x2),int(y2)),(255,0,0),1)


x_center,y_center,width,height = model(image)[0][0][0:4,4849]
x1 = x_center - width/2
y1 = y_center - height/2
x2 = x_center + width/2
y2 = y_center + height/2
img=cv2.imread(file_path)
cv2.rectangle(img,(int(x1),int(y1)),(int(x2),int(y2)),(0,255,0),1)
plt.imshow(img)

In [ ]:
# Load model
model=YOLO("/content/drive/MyDrive/best(1).pt")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# Prepare image
image = image.to(device)
image.requires_grad_(True)
# Store gradients
gradients = {}
def save_grads(key):
    def backward_hook(module, grad_inp, grad_out):
        gradients[key] = grad_out[0]
    return backward_hook


activations = {}

def get_hook(name):
    def hook(module, input, output):
        # detach to avoid storing graph; clone if planning to keep tensor across iterations
        activations[name] = output.cpu()
    return hook
# Pick a layer to attach the hook (example: layer 19)
layer_index = 16
module = model.model.model[layer_index]
module.requires_grad_(True)
module.register_full_backward_hook(save_grads(str(layer_index)))
module.register_forward_hook(get_hook(str(layer_index)))
# Forward pass
output = model.model(image)  # raw feature map output
model.model.zero_grad()
# Choose a scalar for backward pass
o = output[0][0][5][0]  # pick the first element as an example
model.zero_grad(set_to_none=True)
if o.grad is not None:
    o.grad.zero_()
o.backward(retain_graph=True)
def min_max_normalize(x):
    return (x - np.min(x)) / (np.max(x) - np.min(x) + 1e-8)

In [ ]:
gradcam=torch.mean(gradients['16'][0])*activations['16']
heatmap_gradcam=torch.sum(a_gradcam[0],dim=0)
heatmap_gradcam=torch.relu(heatmap_gradcam)

In [ ]:
hirescam=gradients['16'][0]*activations['16'][0]
heatmap_hirescam=torch.sum(a,dim=0)
heatmap_hirescam=torch.relu(heatmap_hirescam)

In [ ]:
heatmap_gradcam_norm = min_max_normalize(heatmap_gradcam)
heatmap_hirescam_norm = min_max_normalize(heatmap_hirescam)

In [ ]:
#heatmap for gradcam
plt.imshow(cv2.imread(file_path))
a=plt.imshow(cv2.resize(heatmap_gradcam_norm.detach().cpu().numpy(),(640,640)),cmap='jet',alpha=0.4)
cbar = plt.colorbar(a)
plt.axis("off")

In [ ]:
#heatmap for hirescam
plt.imshow(cv2.imread(file_path))
a=plt.imshow(cv2.resize(heatmap_hirescam_norm.detach().cpu().numpy(),(640,640)),cmap='jet',alpha=0.4)
cbar = plt.colorbar(a)
plt.axis("off")